# 5. Un micro agent harness completo

Assembliamo un harness autocontenuto con componenti reali LangChain/LangGraph:

- modello OpenAI e prompt di sistema;
- tool tipizzati;
- loop ReAct;
- checkpoint per sospensione e ripresa;
- human-in-the-loop prima di una scrittura;
- limite deterministico delle tool call;
- middleware di audit;
- verifica dell'artefatto indipendente dalla risposta del modello.

Il caso d'uso calcola statistiche e scrive un report solo dopo approvazione.

## Configurazione

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def find_env() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        candidate = directory / '.env'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('.env non trovato.')


ENV_FILE = find_env()
load_dotenv(ENV_FILE, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(f'OPENAI_API_KEY non configurata in {ENV_FILE}')

MODEL_NAME = os.getenv('OPENAI_MODEL') or 'gpt-5.4-mini'
model = ChatOpenAI(
    model=MODEL_NAME,
    reasoning_effort='low',
    use_responses_api=True,
    store=False,
)
print('Modello:', MODEL_NAME)

## Tool con side effect separato dal calcolo

`calculate_statistics` è read-only. `write_report` modifica stato esterno e richiederà approvazione. Separare i due tool permette una policy diversa per rischio.

In [ ]:
import statistics
import tempfile

from langchain.tools import tool

workspace_handle = tempfile.TemporaryDirectory(prefix='complete-harness-')
WORKSPACE = Path(workspace_handle.name).resolve()


@tool
def calculate_statistics(values: list[float]) -> dict[str, float]:
    '''Calcola count, mean, median, minimum e maximum per una lista numerica non vuota.'''
    if not values:
        raise ValueError('La lista non può essere vuota.')
    return {
        'count': float(len(values)),
        'mean': statistics.fmean(values),
        'median': statistics.median(values),
        'minimum': min(values),
        'maximum': max(values),
    }


@tool
def write_report(filename: str, content: str) -> str:
    '''Scrive un report Markdown nel workspace. È un side effect e richiede approvazione.'''
    if Path(filename).name != filename or not filename.endswith('.md'):
        raise ValueError('Usa un semplice nome file con estensione .md.')
    if not content.strip() or len(content) > 12_000:
        raise ValueError('Contenuto vuoto o troppo grande.')
    target = WORKSPACE / filename
    target.write_text(content, encoding='utf-8')
    return f'Report scritto in {target.name} ({len(content)} caratteri).'


print('Workspace:', WORKSPACE)

## Audit hook

Il middleware registra nome, stato e durata. Non conserva argomenti o output: potrebbero contenere dati sensibili.

In [ ]:
import time
from collections.abc import Callable

from langchain.agents.middleware import ToolCallRequest, wrap_tool_call
from langchain_core.messages import ToolMessage
from langgraph.types import Command

audit_events: list[dict[str, object]] = []


@wrap_tool_call
def audit_tool(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    started = time.monotonic()
    event: dict[str, object] = {'tool': request.tool_call['name']}
    try:
        result = handler(request)
        event['status'] = 'ok'
        return result
    except Exception:
        event['status'] = 'error'
        raise
    finally:
        event['elapsed_ms'] = round((time.monotonic() - started) * 1000)
        audit_events.append(event)

## Assemblaggio del graph

`HumanInTheLoopMiddleware` intercetta `write_report` dopo la proposta del modello e prima dell'esecuzione. `InMemorySaver` è necessario perché LangGraph deve salvare il punto esatto da cui riprendere.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
harness = create_agent(
    model=model,
    tools=[calculate_statistics, write_report],
    middleware=[
        audit_tool,
        ToolCallLimitMiddleware(run_limit=8, exit_behavior='end'),
        HumanInTheLoopMiddleware(
            interrupt_on={
                'write_report': {'allowed_decisions': ['approve', 'reject']},
                'calculate_statistics': False,
            }
        ),
    ],
    checkpointer=checkpointer,
    system_prompt=(
        'Sei un agente di reporting. Devi usare calculate_statistics, poi preparare '
        'un report Markdown con dati, interpretazione e limiti, infine chiamare write_report '
        'con filename report.md. Non dichiarare completato finché il tool non conferma la scrittura.'
    ),
)
config = {'configurable': {'thread_id': 'report-demo'}}
print(harness)

## Prima esecuzione: pausa prima del side effect

Il tool di calcolo può essere eseguito. Quando il modello propone `write_report`, il graph restituisce `__interrupt__` e non crea ancora il file.

In [ ]:
initial_result = harness.invoke(
    {
        'messages': [{
            'role': 'user',
            'content': 'Analizza [14, 17, 19, 22, 23, 31] e crea il report richiesto.',
        }]
    },
    config=config,
)

assert '__interrupt__' in initial_result
assert not (WORKSPACE / 'report.md').exists()
interrupt_payload = initial_result['__interrupt__'][0].value
interrupt_payload

## Decisione umana e ripresa

In un'applicazione reale il payload verrebbe mostrato all'utente. Qui lo abbiamo appena ispezionato e approviamo esplicitamente. La ripresa usa lo stesso `thread_id`.

In [ ]:
resumed_result = harness.invoke(
    Command(resume={'decisions': [{'type': 'approve'}]}),
    config=config,
)
print(resumed_result['messages'][-1].text)

## Gate di verifica deterministico

Non accettiamo la frase «ho finito» come prova. Controlliamo filesystem, contenuto, tool eseguiti e audit.

In [ ]:
report_path = WORKSPACE / 'report.md'
report = report_path.read_text(encoding='utf-8')
called_tools = [
    call['name']
    for message in resumed_result['messages']
    for call in (getattr(message, 'tool_calls', None) or [])
]

print(report)
print('Tool proposti:', called_tools)
print('Audit:', audit_events)

assert report_path.is_file()
assert len(report) > 100
assert 'calculate_statistics' in called_tools
assert 'write_report' in called_tools
assert {event['tool'] for event in audit_events} >= {'calculate_statistics', 'write_report'}
assert all('args' not in event and 'output' not in event for event in audit_events)
print('GOAL_COMPLETE: artefatto e prove verificati.')

## Esperimento: rifiutare

Riesegui con un nuovo `thread_id` e riprendi con:

```python
Command(resume={'decisions': [{'type': 'reject', 'message': 'Non autorizzato.'}]})
```

Il middleware sintetizzerà un `ToolMessage` di rifiuto e il file non verrà scritto. Questo mostra la differenza tra una raccomandazione nel prompt e un vincolo applicato dal runtime.

In [ ]:
workspace_handle.cleanup()
print('Workspace temporaneo eliminato.')